# Tree-of-Thought Prompting (ToT)

Generaliza CoT de um caminho linear para uma **árvore de raciocínios paralelos**: o modelo gera k pensamentos candidatos por nível e um avaliador LLM poda os ramos inviáveis antes de descer. **BFS** avalia todos os candidatos de um nível antes de descer — garante a solução mais rasa. **DFS** segue um ramo até o fim com backtracking — menor custo de memória. Yao et al. (2023) reportaram 74% de acerto no 24-game com ToT vs. 4% com CoT padrão.

**Referência:** Yao et al. (2023) *Tree of Thoughts: Deliberate Problem Solving with Large Language Models.* arXiv:2305.10601

In [ ]:
!pip install -q --upgrade langchain-ollama langchain-core python-dotenv langchain requests


In [9]:
from pathlib import Path
import os, json, re, unicodedata
from textwrap import dedent

from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

# ── Configuração local Ollama ──────────────────────────────────────────────
load_dotenv(override=True)
MODEL_NAME = "gemma3:4b"
CREATIVE_MODEL_NAME = "phi4-mini"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")


# ── Constantes ──────────────────────────────────────────────────────────────

# env_candidates = [Path("code/.env"), Path(".env")]
# env_path = next((p for p in env_candidates if p.exists()), None)
# if env_path is None:
#     env_path = Path(".env")

# load_dotenv(dotenv_path=env_path, override=True)

# ── Clientes LLM ───────────────────────────────────────────────────────────
llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
    num_predict=180,
)
llm_criativo = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.4,
    num_predict=120,
)

print(f"✓ Ollama | modelo padrão: {MODEL_NAME}")
print(f"✓ Ollama | modelo gerador: {MODEL_NAME}")
print(f"✓ Ollama | base_url: {OLLAMA_BASE_URL}")
print("Configuração concluída.")

# ── Funções auxiliares ────────────────────────────────────────────────────────
def normalizar(texto: str) -> str:
    """Converte para minúsculas e remove acentos para comparação por palavras-chave."""
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(ch for ch in texto if not unicodedata.combining(ch))

def extrair_json(texto: str) -> dict:
    """Remove cercas Markdown e interpreta o primeiro objeto JSON encontrado."""
    texto = texto.strip()
    if texto.startswith("```"):
        texto = re.sub(r"^```(?:json)?\s*|\s*```$", "", texto, flags=re.S).strip()
    inicio = texto.find("{")
    fim    = texto.rfind("}")
    if inicio != -1 and fim != -1 and fim > inicio:
        texto = texto[inicio : fim + 1]
    return json.loads(texto)

def chamar_texto(llm_client, prompt_template, alternativa: str, **kwargs) -> str:
    """Chama o LLM e retorna texto; usa alternativa em caso de erro."""
    if llm_client is None:
        return alternativa
    try:
        bruto = (prompt_template | llm_client).invoke(kwargs)
        return getattr(bruto, "content", str(bruto)).strip()
    except Exception as exc:
        print(f"⚠ Ollama: {exc}. Usando alternativa.")
        return alternativa

def pedir_json(llm_client, prompt_template, alternativa: dict, **kwargs) -> dict:
    """Chama o LLM e interpreta JSON; usa alternativa em caso de erro."""
    if llm_client is None:
        return alternativa
    try:
        bruto = (prompt_template | llm_client).invoke(kwargs)
        return extrair_json(getattr(bruto, "content", str(bruto)))
    except Exception as exc:
        print(f"⚠ Ollama: {exc}. Usando alternativa.")
        return alternativa


# ── Estado compartilhado ──────────────────────────────────────────────────────────
RAIZ_TOT = "Planejar um sábado chuvoso em Curitiba: 2 adultos, 1 criança, R$120 de orçamento."

# ── Função de avaliação compartilhada ─────────────────────────────────────────────
PROMPT_AVALIAR_PENSAMENTO = PromptTemplate(
    input_variables=["estado", "pensamento"],
    template=(
        "Avalie um nó de Tree-of-Thought.\n"
        "Estado atual: {estado}\n"
        "Pensamento candidato: {pensamento}\n\n"
        "Responda começando com Promissor, Válido ou Inválido.\n"
        "Depois justifique em 1 frase."
    ),
)

def rotular_pensamento(estado: str, pensamento: str) -> str:
    """Rotula um pensamento candidato como Promissor, Válido ou Inválido."""
    return chamar_texto(
        llm,
        PROMPT_AVALIAR_PENSAMENTO,
        alternativa=("Válido. Opção coerente com o estado atual." if "indoor" in normalizar(pensamento)
                  else "Inválido. Não se adapta ao dia chuvoso ou ao orçamento."),
        estado=estado,
        pensamento=pensamento,
    )

def extrair_etiqueta(texto: str) -> str:
    """Extrai a etiqueta Promissor, Válido ou Inválido da resposta do LLM."""
    for etiqueta in ("Promissor", "Válido", "Inválido"):
        if texto.strip().startswith(etiqueta):
            return etiqueta
    primeiros = texto.strip().split()
    return primeiros[0].strip(":-") if primeiros else "Válido"


PROMPT_GERAR_PENSAMENTOS_JSON = PromptTemplate(
    input_variables=["estado", "objetivo", "k"],
    template=(
        "Você está expandindo uma árvore de pensamentos para resolver um problema.\n"
        "Estado atual: {estado}\n"
        "Objetivo: {objetivo}\n\n"
        "Gere exatamente {k} próximos pensamentos curtos, diferentes e acionáveis.\n"
        "Foque em roteiro local de poucas horas; não sugira hospedagem.\n"
        "Não conclua o plano final se ainda houver passos possíveis.\n"
        "Retorne SOMENTE JSON válido neste formato:\n"
        "{{\"pensamentos\": [\"...\", \"...\", \"...\"]}}"
    ),
)

PROMPT_VERIFICAR_SOLUCAO = PromptTemplate(
    input_variables=["estado", "caminho"],
    template=(
        "Avalie se o caminho abaixo já resolve o problema de forma concreta e viável.\n"
        "Problema: {estado}\n"
        "Caminho atual: {caminho}\n\n"
        "Responda SOMENTE JSON válido neste formato:\n"
        "{{\"solucao\": true ou false, \"motivo\": \"frase curta\"}}"
    ),
)


def extrair_lista_pensamentos(texto: str, limite: int) -> list[str]:
    """Extrai pensamentos de JSON ou de lista numerada produzida pelo modelo."""
    try:
        dados = extrair_json(texto)
        pensamentos = dados.get("pensamentos", [])
        if isinstance(pensamentos, list):
            return [str(p).strip() for p in pensamentos if str(p).strip()][:limite]
    except Exception:
        pass

    texto_limpo = re.sub(r"```(?:json)?|```", "", texto, flags=re.I).strip()
    bloco = re.search(r'"pensamentos"\s*:\s*\[(.*?)\]', texto_limpo, flags=re.S)
    if bloco:
        itens = re.findall(r'"([^"\\n]+)"', bloco.group(1))
        if itens:
            return [item.strip() for item in itens if item.strip()][:limite]

    linhas = []
    for linha in texto_limpo.splitlines():
        item = re.sub(r"^\s*(?:\d+[\).:-]|[-*])\s*", "", linha).strip()
        item = item.strip(' ,[]{}"')
        if not item or item.lower().startswith("pensamentos"):
            continue
        linhas.append(item)
    return linhas[:limite]


def gerar_pensamentos(estado: str, objetivo: str, k: int = 3) -> list[str]:
    """Gera os próximos pensamentos da árvore em tempo de execução."""
    pensamentos_alternativos = [
        "Priorizar atrações cobertas e próximas no centro.",
        "Reservar parte do orçamento para alimentação simples.",
        "Evitar deslocamentos longos por causa da chuva.",
    ]
    alternativa = json.dumps({"pensamentos": pensamentos_alternativos}, ensure_ascii=False)
    resposta = chamar_texto(
        llm_criativo,
        PROMPT_GERAR_PENSAMENTOS_JSON,
        alternativa=alternativa,
        estado=estado,
        objetivo=objetivo,
        k=k,
    )
    pensamentos = extrair_lista_pensamentos(resposta, limite=k)
    return pensamentos or pensamentos_alternativos[:k]


def montar_estado_tot(estado_base: str, caminho: list[tuple[str, str]]) -> str:
    """Monta o estado acumulado a partir do caminho já escolhido."""
    if not caminho:
        return estado_base
    etapas = " / ".join(pensamento for pensamento, _ in caminho)
    return f"{estado_base} / {etapas}"


def verificar_solucao(estado_base: str, caminho: list[tuple[str, str]], profundidade_minima: int = 2) -> tuple[bool, str]:
    """Verifica de forma simples se o caminho já resolve o problema de ToT."""
    if len(caminho) < profundidade_minima:
        return False, "Ainda há poucos passos para formar um plano completo."

    texto = normalizar(" ".join(pensamento for pensamento, _ in caminho))
    tem_abrigo = any(palavra in texto for palavra in ("indoor", "cobert", "museu", "livraria", "shopping", "cafe"))
    tem_custo = any(palavra in texto for palavra in ("barat", "econom", "orcamento", "simples", "gratuit", "preco", "acessivel", "baixo", "abaixo", "r$"))
    tem_logistica = any(palavra in texto for palavra in ("proxim", "desloc", "centro", "curta", "caminhar"))

    if tem_abrigo and (tem_custo or tem_logistica):
        return True, "O caminho já combina abrigo contra chuva com custo ou logística viável."
    return False, "O caminho ainda não cobre bem chuva, custo e deslocamento."


print("Configuração ToT concluída.")













✓ Ollama | modelo padrão: gemma3:4b
✓ Ollama | modelo gerador: gemma3:4b
✓ Ollama | base_url: http://172.18.224.1:11434
Configuração concluída.
Configuração ToT concluída.


## 01. Geração de Pensamentos

O modelo gera múltiplas opções **antes** de decidir — abre o espaço de busca em vez de seguir um único caminho.

In [10]:
# ── 01. Geração de Pensamentos ────────────────────────────────────────────
# O modelo gera múltiplas opções ANTES de decidir o próximo passo.
# Isso abre o espaço de busca em vez de seguir um único caminho.

ESTADO_TOT = RAIZ_TOT
OBJETIVO_TOT = "Gerar próximos passos para um plano coberto, barato e viável."

pensamentos_gerados = gerar_pensamentos(
    ESTADO_TOT,
    OBJETIVO_TOT,
    k=2,
)

print("Estado:")
print(ESTADO_TOT)
print("\nPensamentos gerados:")
for i, pensamento in enumerate(pensamentos_gerados, 1):
    print(f"{i}. {pensamento}")




Estado:
Planejar um sábado chuvoso em Curitiba: 2 adultos, 1 criança, R$120 de orçamento.

Pensamentos gerados:
1. Pesquisar museus ou centros culturais em Curitiba com entrada gratuita ou de baixo custo.
2. Identificar parques cobertos ou espaços públicos com áreas de descanso que possam oferecer abrigo da chuva.


## 02. Avaliação de Estados

Um avaliador poda ramos ruins e mantém os promissores — evita desperdiçar chamadas em direções sem saída.

In [3]:
# ── 02. Avaliação de Estados ──────────────────────────────────────────────
# Um avaliador poda ramos ruins (Inválido) e mantém os promissores.
# Isso evita desperdiçar chamadas em direções sem saída.

CANDIDATOS_NIVEL1 = [
    "Focar em atrações indoor no centro e caminhar pouco.",
    "Começar com parque ao ar livre e esperar o tempo melhorar.",
    "Montar roteiro de shopping com livros, café e atividade infantil.",
]

print("Estado-base:")
print(RAIZ_TOT)
print("\nAvaliação dos candidatos:")
for pensamento in CANDIDATOS_NIVEL1:
    resposta = rotular_pensamento(RAIZ_TOT, pensamento)
    etiqueta = extrair_etiqueta(resposta)
    print(f"\n[{etiqueta}] {pensamento}")
    print(f"  {resposta}")


Estado-base:
Planejar um sábado chuvoso em Curitiba: 2 adultos, 1 criança, R$120 de orçamento.

Avaliação dos candidatos:

[Válido] Focar em atrações indoor no centro e caminhar pouco.
  Válido.

A sugestão de focar em atrações indoor no centro e minimizar o caminhar é uma estratégia sensata para um sábado chuvoso em Curitiba com um orçamento limitado e a presença de uma criança.

[Inválido] Começar com parque ao ar livre e esperar o tempo melhorar.
  Inválido. A sugestão de começar com parques ao ar livre é inadequada considerando que o sábado é chuvoso em Curitiba, tornando a atividade impraticável.

[Válido] Montar roteiro de shopping com livros, café e atividade infantil.
  Válido.

O pensamento candidato é um passo lógico e relevante para o planejamento do sábado, focando em atividades e opções dentro do orçamento e do contexto da situação.


## 03. BFS — Busca em Largura

Avalia **todos** os candidatos de um nível antes de descer. Mantém o top-k mais promissor para o próximo nível.

In [14]:
# ── 03. BFS — Busca em Largura com ToT dinâmico ───────────────────────────
# BFS avalia todos os caminhos da fronteira atual antes de descer de nível.
# Aqui a árvore não é fixa: cada expansão gera novos pensamentos com o LLM.

PRIORIDADE_BFS = {"Promissor": 3, "Válido": 2, "Inválido": 1}
OBJETIVO_BUSCA_TOT = "Construir um plano final para o dia chuvoso, com baixo custo e atrações cobertas."
LARGURA_BFS = 2
PROFUNDIDADE_MAX_BFS = 3

fronteira = [[]]
solucao_bfs = None

for profundidade in range(PROFUNDIDADE_MAX_BFS):
    print(f"BFS — Nível {profundidade + 1}:")
    candidatos_nivel = []

    for caminho in fronteira:
        estado_atual = montar_estado_tot(RAIZ_TOT, caminho)
        pensamentos = gerar_pensamentos(estado_atual, OBJETIVO_BUSCA_TOT, k=2)

        print(f"\n  Expandindo caminho: {' -> '.join(p for p, _ in caminho) or 'raiz'}")
        for pensamento in pensamentos:
            resposta = rotular_pensamento(estado_atual, pensamento)
            etiqueta = extrair_etiqueta(resposta)
            print(f"    [{etiqueta}] {pensamento}")

            if etiqueta == "Inválido":
                print("      → Poda: candidato inválido.")
                continue

            novo_caminho = [*caminho, (pensamento, etiqueta)]
            eh_solucao, motivo = verificar_solucao(RAIZ_TOT, novo_caminho, profundidade_minima=2)
            candidatos_nivel.append((novo_caminho, etiqueta, motivo))

            if eh_solucao and solucao_bfs is None:
                solucao_bfs = (novo_caminho, motivo)
                break
        if solucao_bfs is not None:
            break

    if solucao_bfs is not None:
        break

    candidatos_nivel = sorted(
        candidatos_nivel,
        key=lambda item: PRIORIDADE_BFS.get(item[1], 0),
        reverse=True,
    )
    fronteira = [caminho for caminho, _, _ in candidatos_nivel[:LARGURA_BFS]]

    print("\n  Fronteira mantida para o próximo nível:")
    for caminho in fronteira:
        print("   - " + " -> ".join(p for p, _ in caminho))

    if not fronteira:
        break

if solucao_bfs is None:
    print("\nBFS não encontrou solução dentro do limite de profundidade.")
else:
    caminho, motivo = solucao_bfs
    print("\nCaminho BFS final:")
    for i, (pensamento, etiqueta) in enumerate(caminho, 1):
        prefixo = "  " if i == 1 else "  → "
        print(f"{prefixo}{pensamento} [{etiqueta}]")
    print(f"Motivo: {motivo}")





BFS — Nível 1:

  Expandindo caminho: raiz
    [Válido] Pesquisar museus e centros culturais em Curitiba com entrada gratuita ou de baixo custo.
    [Válido] Verificar se há eventos culturais internos (ex: palestras, oficinas) em espaços públicos que sejam gratuitos ou de baixo custo.

  Fronteira mantida para o próximo nível:
   - Pesquisar museus e centros culturais em Curitiba com entrada gratuita ou de baixo custo.
   - Verificar se há eventos culturais internos (ex: palestras, oficinas) em espaços públicos que sejam gratuitos ou de baixo custo.
BFS — Nível 2:

  Expandindo caminho: Pesquisar museus e centros culturais em Curitiba com entrada gratuita ou de baixo custo.
    [Válido] Pesquisar o Museu Oscar Niemeyer (MON) em Curitiba, verificando a disponibilidade de visitas guiadas gratuitas ou horários de entrada gratuita.

Caminho BFS final:
  Pesquisar museus e centros culturais em Curitiba com entrada gratuita ou de baixo custo. [Válido]
  → Pesquisar o Museu Oscar Niemeyer (MO

## 04. DFS — Busca em Profundidade

Segue um ramo gerado pelo modelo até o fim; se o ramo não gerar solução, faz **backtracking real** e tenta o próximo pensamento disponível no nível anterior.


In [15]:
# ── 04. DFS — Busca em Profundidade com ToT dinâmico ──────────────────────
# DFS gera pensamentos em cada estado, tenta o primeiro candidato viável e desce.
# Se o ramo falha, volta ao nível anterior e testa o próximo pensamento.

ESTADO_DFS = "Planejar um domingo chuvoso em Curitiba: 2 adultos, 1 criança, R$120."
OBJETIVO_DFS = "Construir um plano final para o dia chuvoso, com baixo custo e atrações cobertas."
PROFUNDIDADE_MAX_DFS = 3


def buscar_dfs_tot(
    estado_base: str,
    caminho: list[tuple[str, str]] | None = None,
    profundidade: int = 0,
) -> tuple[list[tuple[str, str]], str] | None:
    """Executa DFS em uma árvore de pensamentos gerada dinamicamente pelo LLM."""
    caminho = caminho or []
    recuo = "  " * profundidade

    if profundidade >= PROFUNDIDADE_MAX_DFS:
        eh_solucao, motivo = verificar_solucao(estado_base, caminho, profundidade_minima=2)
        if eh_solucao:
            print(f"{recuo}✓ Limite atingido com solução: {motivo}")
            return caminho, motivo
        print(f"{recuo}→ Backtrack: limite atingido sem solução.")
        return None

    estado_atual = montar_estado_tot(estado_base, caminho)
    pensamentos = gerar_pensamentos(estado_atual, OBJETIVO_DFS, k=2)

    if not pensamentos:
        print(f"{recuo}→ Backtrack: nenhum pensamento gerado.")
        return None

    print(f"{recuo}DFS — Nível {profundidade + 1}:")
    for pensamento in pensamentos:
        resposta = rotular_pensamento(estado_atual, pensamento)
        etiqueta = extrair_etiqueta(resposta)
        print(f"{recuo}  [{etiqueta}] {pensamento}")

        if etiqueta == "Inválido":
            print(f"{recuo}    → Poda: candidato inválido.")
            continue

        novo_caminho = [*caminho, (pensamento, etiqueta)]
        eh_solucao, motivo = verificar_solucao(estado_base, novo_caminho, profundidade_minima=2)
        if eh_solucao:
            print(f"{recuo}    ✓ Solução encontrada: {motivo}")
            return novo_caminho, motivo

        resultado = buscar_dfs_tot(
            estado_base,
            caminho=novo_caminho,
            profundidade=profundidade + 1,
        )
        if resultado is not None:
            return resultado

        print(f"{recuo}    → Backtrack: testando próximo candidato do nível {profundidade + 1}.")

    return None


resultado_dfs = buscar_dfs_tot(ESTADO_DFS)

if resultado_dfs is None:
    print("\nNenhum caminho DFS válido foi encontrado.")
else:
    caminho_final, motivo_final = resultado_dfs
    print("\nCaminho DFS final:")
    for i, (pensamento, etiqueta) in enumerate(caminho_final, 1):
        prefixo = "  " if i == 1 else "  → "
        print(f"{prefixo}{pensamento} [{etiqueta}]")
    print(f"Motivo: {motivo_final}")





DFS — Nível 1:
  [Válido] Pesquisar museus e centros culturais em Curitiba com entrada gratuita ou de baixo custo.
  DFS — Nível 2:
    [Válido] Pesquisar o Museu Oscar Niemeyer (MON) em Curitiba - verificar se há exposições gratuitas e horários de funcionamento.
      ✓ Solução encontrada: O caminho já combina abrigo contra chuva com custo ou logística viável.

Caminho DFS final:
  Pesquisar museus e centros culturais em Curitiba com entrada gratuita ou de baixo custo. [Válido]
  → Pesquisar o Museu Oscar Niemeyer (MON) em Curitiba - verificar se há exposições gratuitas e horários de funcionamento. [Válido]
Motivo: O caminho já combina abrigo contra chuva com custo ou logística viável.
